In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

analysis_df = pd.read_csv("synthetic_esbl_data.csv")
#remove n_sites and n_samples columns if they exist
analysis_df = analysis_df.loc[:, ~analysis_df.columns.isin(['n_sites', 'n_samples'])]
analysis_df['prophylaxis_group'] = analysis_df['prophylaxis_group'].replace(
    {'co-amoxiclav (contains penicillin)' : 'amoxicillin_clavulanate'})

In [2]:
analysis_df.columns

Index(['age_at_admission', 'n_surgeries', 'imd_decile', 'past_abx', 'anemia',
       'asthma', 'cancer', 'copd', 'hypertension', 'ischaemic_heart_disease',
       'obesity', 'renal_failure', 'type2_diabetes', 'gender', 'organism_bug',
       'site', 'tfc_group', 'prophylaxis_group', 'ethnicity_desc',
       'esbl_status'],
      dtype='str')

In [3]:
# Load datasets

aware_df = pd.read_excel('WHO_aware.xlsx')

aware_df.columns = aware_df.columns.str.strip()  # Remove leading/trailing whitespace from column names
aware_df = aware_df[['Antibiotic', 'Class', 'Category']]
aware_df['aware_category'] = aware_df['Category'].map({
    'Access': 1,
    'Watch': 2,
    'Reserve': 3
})

asi_df = pd.read_csv('asi_lookup.csv')

aware_df['Antibiotic'] = (aware_df['Antibiotic']
                          .str.strip()
                          .str.split('_')
                          .str[0]
                          .str.replace('Amoxicillin/clavulanic-acid', 'amoxicillin_clavulanate', regex=False))

In [4]:
asi_df['Drug']

0                     Dicloxacillin
1                         Oxacillin
2                       Amoxicillin
3                        Ampicillin
4                        Cephalexin
5                      Erythromycin
6                     Metronidazole
7                        Penicillin
8                         Aztreonam
9                         Cefazolin
10                         Cefdinir
11                         Cefixime
12                      Cefpodoxime
13                         Rifampin
14                     Azithromycin
15                        Cefprozil
16                      Ceftazidime
17                       Cefuroxime
18                  Chloramphenicol
19                   Clarithromycin
20                      Clindamycin
21                     Piperacillin
22    Trimethoprim_sulfamethoxazole
23                       Cefotaxime
24                        Cefoxitin
25                      Ceftriaxone
26                   Colistimethate
27                       Dap

In [5]:
def get_asi(antibiotic_string):
    all_abx = (antibiotic_string
           .replace(" ", "")
           .split("|"))
    
    asi_val = 0
    for abx in all_abx:
        if abx in asi_df["Drug"].str.lower().values:
            asi_val += asi_df.loc[asi_df["Drug"].str.lower() == abx, "Antibiotic_Spectrum_Index"].values[0]
        elif abx == "no_prophylaxis":
            return 0
        elif abx == "teicoplanin":
            asi_val += asi_df.loc[asi_df["Drug"].str.lower() == "vancomycin", "Antibiotic_Spectrum_Index"].values[0]
        else:
            print(f"Warning: Antibiotic '{abx}' not found in ASI dataframe. Skipping.")
    return asi_val

def get_gram_negative_coverage(antibiotic_string):

    gram_negative_cols = [
    "Moraxella_Haemophilus",
    "E_coli_Klebsiella",
    "Enterobacter_Serratia_Citrobacter",
    "ESBL",
    "Pseudomonas"
]

    all_abx = (antibiotic_string
           .replace(" ", "")
           .split("|"))
    
    for abx in all_abx:
        if asi_df.loc[asi_df["Drug"].str.lower() == abx, gram_negative_cols].values.sum() > 0:
            return 1
        elif abx == "no_prophylaxis":
            return 0
        elif abx == "teicoplanin":
            if asi_df.loc[asi_df["Drug"].str.lower() == "vancomycin", gram_negative_cols].values.sum() > 0:
                return 1
    return 0

def get_aware_category(antibiotic_string):
    
    all_abx = (antibiotic_string
           .replace(" ", "")
           .split("|"))
    
    aware_category = 0

    for abx in all_abx:
        if abx in aware_df["Antibiotic"].str.lower().values:
            category_val = aware_df.loc[aware_df["Antibiotic"].str.lower() == abx, "aware_category"].values[0]
            aware_category = max(aware_category, category_val)
        elif abx == "no_prophylaxis":
            return 0
        else:
            print(f"Warning: Antibiotic '{abx}' not found in AWARE dataframe. Skipping.")
    return aware_category
    
analysis_df['asi_value'] = analysis_df['prophylaxis_group'].fillna('no_prophylaxis').apply(get_asi)
analysis_df['gram_neg_coverage'] = analysis_df['prophylaxis_group'].fillna('no_prophylaxis').apply(get_gram_negative_coverage)

analysis_df['aware'] = analysis_df['prophylaxis_group'].fillna('no_prophylaxis').apply(get_aware_category)